*Title:*: 4_Repeatability
*Goal:* Generate the repeatability type of each SNP

# 4.1_Get counts 
FIRST NEED TO MANUALLY REMOVE THE SECOND ROW (EMPTY):

``` bash
awk 'BEGIN {
    FS = OFS = "\t" # Set field separator (FS) and output field separator (OFS) to tab
    comb[1] = "2 8" # Define row combinations
    comb[2] = "3 9"
    comb[3] = "4 10"
    comb[4] = "5 11"
    comb[5] = "6 12"
    comb[6] = "7 13"
    comb[7] = "2 3 4 5 6 7"
    comb[8] = "8 9 10 11 12 13"
    names[1] = "tYounger" # Define category names
    names[2] = "tOldDairy"
    names[3] = "tScott"
    names[4] = "tLombardi"
    names[5] = "tLaguna"
    names[6] = "tWaddell"
    names[7] = "spatial_2016"
    names[8] = "spatial_2017"
}
NR == 1 {
    split($0, headers, FS) # Split the header line into an array
    next
}
{
    for (i = 1; i <= 8; i++) { # Loop over combinations
        split(comb[i], rows, " ") # Split the combination into row indices
        for (j = 1; j <= NF; j++) { # Loop over fields in the current line
            for (k in rows) { # Loop over row indices
                if (NR == rows[k] && $j == "TRUE") { # Check row and value
                    sum[i, j]++ # Increment count for this combination and field
                }
            }
        }
    }
    for (j = 1; j <= NF; j++) { # Loop over fields in the current line
        total[j] += ($j == "TRUE") # Increment total count for each field if value is "TRUE"
    }
}
END {
    print "ColumnNumber", "Count", "tYounger", "tOldDairy", "tScott", "tLombardi", "tLaguna", "tWaddell", "spatial_2016", "spatial_2017"
    for (j = 1; j <= NF; j++) {
        printf "%s\t%d", headers[j], total[j] # Print the header instead of column number
        for (i = 1; i <= 8; i++) printf "\t%d", sum[i, j] + 0
        print ""
    }
}' /home/mary2018/scratch/MultiYear/analysis/masked/REDO/poolfstat-fst-version-ii/upset/4.outliers_poolfstat_SNP_FST_95_2016_2017_outliers_only_AGAIN.tsv  > repeatability_AGAIN.tsv
``` 

# 4.2. Clean/Fix in R
output file: repeatability.tsv

*Goal:* This file will contain the number of times a SNP is an outlier in each estuary, and in each year. Here, adding columns for pure spatial counts (i.e. counts if only an outlier in 2016 and only an outlier in 2017), also dividing the counts by 2 and removing the rows where the count is 1. This is because the counts are doubled in the upset script, and we only want to keep the SNPs that are outliers in both years.
- i.e. I want a tLBD to mean that it is a temporal outlier in LBD, that that it is an outlier for LBD in either 2016 or 2017. Currently, the script is just counting T for LBD 2016 and 2017, so a count of 2 actually means temporally repeated. Below script fixes that!


- ColumnNumber: The column number in the input file
- Count: The total number of times the SNP is an outlier
- tYounger: The number of times the SNP is an outlier in the Younger estuary
- tOldDairy: The number of times the SNP is an outlier in the Old Dairy estuary
- tScott: The number of times the SNP is an outlier in the Scott estuary
- tLombardi: The number of times the SNP is an outlier in the Lombardi estuary
- tLaguna: The number of times the SNP is an outlier in the Laguna estuary
- tWaddell: The number of times the SNP is an outlier in the Waddell estuary
- spatial-2016: The number of times the SNP is an outlier in 2016
- spatial-2017: The number of times the SNP is an outlier in 2017
- spatial_temporal_count= Means the number of times (estuaries) that a given SNP is an outlier in both years.
- only_2016_spatial: The number of times the SNP is an outlier in 2016, but not in 2017
- only_2017_spatial: The number of times the SNP is an outlier in 2017, but not in 2016

``` R
library(data.table)
library(tidyverse)
library(ggplot2)


file<- "repeatability.tsv"
data<- read.table(file, header=TRUE, sep="\t")
data<-data[-1,] #removing extra row
data$only_2016_spatial<- ifelse (data$spatial_2017 == 0, data$spatial_2016, 0) #making a column for pure spatial 2016 (not in 2017 at all). If not pure to 2016, a 0 is printe.d
data$only_2017_spatial<- ifelse (data$spatial_2016 == 0, data$spatial_2017, 0)  #making a column for pure spatial 2017 (not in 2016 at all)
columns_to_modify <- c("tYounger", "tOldDairy", "tScott", "tLombardi", "tLaguna", "tWaddell")
data[columns_to_modify] <- lapply(data[columns_to_modify], function(x) x/2)
data[columns_to_modify] <- lapply(data[columns_to_modify], function(x) ifelse(x == 0.5, 0, x)) #removing partial temporals
data$spatial_temporal_count <- rowSums(data[columns_to_modify])

#Make local counts:
data$Local_Younger <- ifelse(data$tYounger == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_OldDairy <- ifelse(data$tOldDairy == 1 & data$tYounger == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Waddell <- ifelse(data$tWaddell == 1 & data$tOldDairy == 0 & data$tYounger == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Laguna <- ifelse(data$tLaguna == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tYounger == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Lombardi <- ifelse(data$tLombardi == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tYounger == 0 & data$tScott == 0, 1, 0)

data$Local_Scott <- ifelse(data$tScott == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tYounger == 0, 1, 0)


#save output as .tsv file:
write.table(data, file="repeatability_cleaned_AGAIN.tsv", sep="\t", row.names=FALSE, quote=FALSE)



#Gathering and saving results:
spatial_temporal_counts<- as.data.frame(table(data$spatial_temporal_count))
only_2016_spatial_counts<- as.data.frame(table(data$only_2016_spatial))
only_2017_spatial_counts<- as.data.frame(table(data$only_2017_spatial))
spatial_2016_counts<- as.data.frame(table(data$spatial_2016))
spatial_2017_counts<-as.data.frame(table(data$spatial_2017))

#Changing the column maes for later merging:
colnames(spatial_temporal_counts) <- c("Number_of_estuaries", "spatial_temporal_counts")
colnames(only_2016_spatial_counts) <- c("Number_of_estuaries", "only_2016_spatial_counts")
colnames(only_2017_spatial_counts) <- c("Number_of_estuaries", "only_2017_spatial_counts")
colnames(spatial_2016_counts) <- c("Number_of_estuaries", "spatial_2016_counts")
colnames(spatial_2017_counts) <- c("Number_of_estuaries", "spatial_2017_counts")


summary_repeatabililty_counts <- Reduce(function(x, y) merge(x, y, by = "Number_of_estuaries", all = TRUE), 
                                        list(spatial_temporal_counts, only_2016_spatial_counts, 
                                             only_2017_spatial_counts, spatial_2016_counts, 
                                             spatial_2017_counts))

#Replace NA with 0:
summary_repeatabililty_counts[is.na(summary_repeatabililty_counts)] <- 0
#Save as .tsv file:
write.table(summary_repeatabililty_counts, file="summary_repeatabililty_counts_AGAIN.tsv", sep="\t", row.names=FALSE, quote=FALSE)
``` 